# 🏷️ Notebook 07 — BioBERT Classification Model
**Healthcare RAG-Powered Medical Q&A Assistant**
**eyouth × DEPI | Microsoft Machine Learning Track | 2026**

---

### 🎯 Objectives
- Fine-tune `dmis-lab/biobert-v1.1 (BioBERT)` on 6 medical categories
- Use 80/10/10 train/val/test split
- Apply class weights to handle category imbalance
- Evaluate with per-class F1, macro F1, and accuracy
- Target: macro F1 ≥ 78%
- Save model to `models/classifier/biobert_classifier/`

### 📋 Deliverables
- `notebooks/07_classification_model.ipynb`
- `models/classifier/biobert_classifier/` (saved model)
- Classification report saved to `reports/`

---

## 1. Imports & Setup

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

os.makedirs('../models/classifier', exist_ok=True)
os.makedirs('../reports/figures', exist_ok=True)
os.makedirs('../reports', exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Libraries loaded | Device: {device}")

## 2. Load & Prepare Data

In [ ]:
df = pd.read_csv('../data/processed/pubmedqa_labelled.csv')

# Build input text: question + context for richer signal
# BioBERT is a cased model — preserve original capitalisation
df['text'] = df['question'].astype(str) + " [SEP] " + df['context'].astype(str)

print(f"Dataset shape: {df.shape}")
print(f"\nCategory distribution:")
print(df['category'].value_counts())

# Label mapping (sorted for consistency)
label2id = {label: idx for idx, label in enumerate(sorted(df['category'].unique()))}
id2label = {idx: label for label, idx in label2id.items()}

df['label_id'] = df['category'].map(label2id)

print(f"\nLabel mapping: {label2id}")
print(f"Number of classes: {len(label2id)}")

## 3. Compute Class Weights

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(label2id)),
    y=df['label_id'].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print("Class weights:")
for label, weight in zip(sorted(label2id.keys()), class_weights):
    print(f"  {label:<15} → {weight:.4f}")

## 4. Train / Validation / Test Split (80/10/10)

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df['label_id'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label_id'], random_state=42
)

print(f"Train size: {len(train_df):,} ({len(train_df)/len(df)*100:.0f}%)")
print(f"Val size:   {len(val_df):,} ({len(val_df)/len(df)*100:.0f}%)")
print(f"Test size:  {len(test_df):,} ({len(test_df)/len(df)*100:.0f}%)")

## 5. Tokenizer & Dataset

BioBERT (`dmis-lab/biobert-v1.1`) is pre-trained on PubMed + PMC text — significantly better domain fit than general-purpose DistilBERT for medical text classification. It is a **cased** model — inputs must **not** be lowercased.

In [ ]:
MODEL_NAME      = "dmis-lab/biobert-v1.1"
LOCAL_SAVE_PATH = "../models/classifier/biobert_classifier"
HF_REPO_ID      = "AbdoMatrix/biobert-medical-classifier"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Loaded tokenizer: {MODEL_NAME} (cased)")

In [ ]:
class MedicalDataset(Dataset):
    """PyTorch Dataset wrapping tokenised BioBERT inputs."""

    def __init__(self, texts: list, labels: list, tokenizer, max_length: int = 256):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


train_dataset = MedicalDataset(
    train_df["text"].tolist(), train_df["label_id"].tolist(), tokenizer
)
val_dataset = MedicalDataset(
    val_df["text"].tolist(), val_df["label_id"].tolist(), tokenizer
)
test_dataset = MedicalDataset(
    test_df["text"].tolist(), test_df["label_id"].tolist(), tokenizer
)

print(f"✅ Datasets created")
print(f"   Train: {len(train_dataset):,} samples")
print(f"   Val:   {len(val_dataset):,} samples")
print(f"   Test:  {len(test_dataset):,} samples")

## 6. Custom Trainer with Weighted Loss

HuggingFace `Trainer` doesn't use class weights by default.
We override `compute_loss` to apply weighted cross-entropy.

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

print("✅ WeightedTrainer defined")

## 7. Model & Training Configuration

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir='../models/classifier/checkpoints',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
    fp16=torch.cuda.is_available(),
)


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    f1_macro    = f1_score(labels, preds, average='macro')
    f1_weighted = f1_score(labels, preds, average='weighted')
    acc         = accuracy_score(labels, preds)
    return {
        'f1_macro':    f1_macro,
        'f1_weighted': f1_weighted,
        'accuracy':    acc,
    }


print(f"✅ Model initialized | Epochs: 3 | LR: 2e-5 | Batch: 16")

## 8. Train

In [ ]:
trainer = WeightedTrainer(
    class_weights=class_weights_tensor,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("🚀 Starting fine-tuning...")
trainer.train()

## 9. Save Model Locally

In [ ]:
os.makedirs(LOCAL_SAVE_PATH, exist_ok=True)
trainer.save_model(LOCAL_SAVE_PATH)
tokenizer.save_pretrained(LOCAL_SAVE_PATH)
print(f"✅ Model saved locally: {LOCAL_SAVE_PATH}")

## 10. Evaluate on Test Set

In [ ]:
predictions = trainer.predict(test_dataset)
preds       = np.argmax(predictions.predictions, axis=1)
true_labels = test_df['label_id'].values

# ── Classification Report ────────────────────────────────────────────────────
report_str = classification_report(
    true_labels, preds,
    target_names=sorted(label2id.keys())
)

print("=" * 60)
print("CLASSIFICATION REPORT (Test Set)")
print("=" * 60)
print(report_str)

# ── Key Metrics ──────────────────────────────────────────────────────────────
macro_f1    = f1_score(true_labels, preds, average='macro')
weighted_f1 = f1_score(true_labels, preds, average='weighted')
acc         = accuracy_score(true_labels, preds)

print(f"\n🎯 Macro F1:    {macro_f1:.4f}")
print(f"🎯 Weighted F1: {weighted_f1:.4f}")
print(f"🎯 Accuracy:    {acc:.4f}")

if macro_f1 >= 0.78:
    print("\n✅ KPI MET: Macro F1 ≥ 78%")
else:
    print(f"\n⚠️  KPI NOT MET: Macro F1 = {macro_f1:.4f} (target ≥ 0.78)")
    print("    Consider: more epochs, different learning rate, or data augmentation")

## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(true_labels, preds)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=sorted(label2id.keys()),
    yticklabels=sorted(label2id.keys())
)
plt.title(f'Confusion Matrix — Macro F1: {macro_f1:.4f}', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig('../reports/figures/07_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## 12. Save Classification Report & Upload to HuggingFace

In [ ]:
from huggingface_hub import HfApi

# ── Save classification report ───────────────────────────────────────────────
report_path = "../reports/classification_report.md"

report_content = f"""# Classification Report — BioBERT Medical Classifier

## Model Details
| Item | Value |
|---|---|
| Base model | `dmis-lab/biobert-v1.1` |
| Classes | {len(label2id)} |
| Training split | 80/10/10 |
| Epochs | 3 |
| Learning rate | 2e-5 |
| Batch size | 16 |
| Class weights | Applied (balanced) |

## Test Set Results
{report_str}

## Key Metrics
| Metric | Value |
|---|---|
| Macro F1 | {macro_f1:.4f} |
| Weighted F1 | {weighted_f1:.4f} |
| Accuracy | {acc:.4f} |
| KPI (Macro F1 ≥ 0.78) | {"✅ MET" if macro_f1 >= 0.78 else "⚠️ NOT MET"} |

## Label Mapping
{chr(10).join(f"| {label} | {idx} |" for label, idx in sorted(label2id.items()))}
"""

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_content)
print(f"✅ Classification report saved to: {report_path}")

# ── Build model card ─────────────────────────────────────────────────────────
category_rows = "\n".join(f"| {idx} | {label} |" for idx, label in sorted(id2label.items()))

usage_code = (
    'from transformers import AutoTokenizer, AutoModelForSequenceClassification\n'
    'import torch\n\n'
    f'tokenizer = AutoTokenizer.from_pretrained("{HF_REPO_ID}")\n'
    f'model = AutoModelForSequenceClassification.from_pretrained("{HF_REPO_ID}")\n\n'
    'text = "What are the symptoms of diabetes?"\n'
    'inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)\n\n'
    'with torch.no_grad():\n'
    '    outputs = model(**inputs)\n\n'
    'predicted = model.config.id2label[torch.argmax(outputs.logits, dim=1).item()]\n'
    'print(predicted)  # → Symptoms'
)

model_card = f"""---
language: en
license: mit
tags:
  - medical
  - classification
  - biobert
  - pubmedqa
  - healthcare-rag
datasets:
  - qiaojin/PubMedQA
metrics:
  - f1
pipeline_tag: text-classification
---

# BioBERT Medical Query Classifier

Fine-tuned `dmis-lab/biobert-v1.1` for classifying medical questions into 6 categories.

## Categories
| ID | Category |
|----|----------|
{category_rows}

## Results
| Metric | Score |
|--------|-------|
| Macro F1 | {macro_f1:.4f} |
| Weighted F1 | {weighted_f1:.4f} |
| Accuracy | {acc:.4f} |

## Training Config
| Item | Value |
|------|-------|
| Base model | dmis-lab/biobert-v1.1 |
| Dataset | qiaojin/PubMedQA ({len(df):,} rows) |
| Split | 80/10/10 |
| Epochs | 3 |
| Learning rate | 2e-5 |
| Batch size | 16 |
| Class weights | Balanced (custom WeightedTrainer) |

## Usage
{usage_code}

## Project
Healthcare RAG-Powered Medical Q&A Assistant
eyouth x DEPI | Microsoft Machine Learning Track | 2026
GitHub: https://github.com/AbdooMatrix/Healthcare-RAG-Powered-Medical-QA-Assistant
"""

card_path = os.path.join(LOCAL_SAVE_PATH, "README.md")
with open(card_path, "w", encoding="utf-8") as f:
    f.write(model_card)
print(f"✅ Model card saved: {card_path}")

# ── Upload to HuggingFace ────────────────────────────────────────────────────
try:
    api = HfApi()
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=LOCAL_SAVE_PATH, repo_id=HF_REPO_ID, repo_type="model")
    print(f"\n✅ Model uploaded to HuggingFace: https://huggingface.co/{HF_REPO_ID}")
except Exception as e:
    print(f"\n⚠️  HuggingFace upload failed: {e}")
    print("   Model is saved locally. Upload manually later with:")
    print(f"   huggingface-cli upload {HF_REPO_ID} {LOCAL_SAVE_PATH}")


## 13. Quick Test — Verify Saved Model Loads

In [ ]:
sys.path.append(os.path.abspath('..'))
from src.classification.classifier import load_classifier

clf = load_classifier()

test_texts = [
    "What are the symptoms of diabetes?",
    "How is pneumonia diagnosed?",
    "What is the treatment for hypertension?",
    "What are the side effects of aspirin?",
    "How can heart disease be prevented?",
    "What is the role of antibodies?",
]

print("\nQuick classification test:")
for text in test_texts:
    cat = clf.predict(text)
    print(f"  '{text[:50]}...' → {cat}")

## ✅ Summary

| Item | Status |
|---|---|
| Model | `dmis-lab/biobert-v1.1 (BioBERT)` fine-tuned |
| Split | 80/10/10 |
| Epochs | 3 |
| Class weights | Applied via custom WeightedTrainer |
| Macro F1 | Evaluated and logged |
| Accuracy | Evaluated and logged |
| Model saved | `models/classifier/biobert_classifier/` |
| Report saved | `reports/classification_report.md` |

---

### ➡️ Next Step
Open **`08_evaluation.ipynb`** to evaluate RAG vs plain LLM.